# PRO3342 Modelagem e Simulação de Sistemas de Produção
## Projeto BVE - Modelo Conceitual (modelo)

### Consultoria [nome fantasia] - Grupo G112

Francisco Rocha Freitas Brandão 18404341
Vasco Oliveira 18440540
Inigo Anduaga 18404466
Kepa Danis 18404719

**Versão:** [1.0] | **Data:** [dd/mm/aaaa]

| Versão | Data | Alterações |
|:--|:--|:--|
| 1.0 | [dd/mm/aaaa] | Versão inicial |
| 1.1 | [11/10/2026] | Segunda Versão

> **Como usar este documento.** Este é um modelo para o documento que especifica a operação de bunkering da BVE. O modelo conceitual descreve o que será representado no modelo computacional, com precisão suficiente para que outra pessoa possa implementá-lo. É um documento técnico interno da consultoria, não um documento comercial. Como ele evolui ao longo do projeto, registre as alterações na tabela de versões acima.
>
> O objetivo, o escopo e os indicadores do estudo já foram definidos na proposta. Cite-os quando necessário, sem repetir o texto.
>
> **Requisitos mínimos.** O documento precisa especificar: os elementos do sistema e seus atributos; os quatro processos e suas interações; as regras de decisão do PCO; os eventos e as variáveis de estado; os parâmetros e indicadores; as premissas adotadas.
>
> **Estrutura.** As seções abaixo são uma sugestão. Use tabelas, fluxogramas e diagramas sempre que forem mais claros que o texto corrido. No Markdown, figuras são inseridas com `![](caminho/figura.png)`.
>
> As orientações em citação, como esta, devem ser apagadas na versão final.

## 1. Elementos do sistema

> Liste os elementos do sistema com seus atributos e sua capacidade. Considere navios, pedidos de abastecimento, barcaças, base de carregamento da BVE, terminais do porto, canal e fundeio e o PCO. Para os pedidos, considere atributos como volume, local de atendimento e janela de atendimento. Atributos são as informações que cada elemento carrega e que o modelo precisa consultar ou atualizar.

| Elemento | Tipo | Atributos | Quantidade / capacidade |
|:--|:--|:--|:--|
| [Navio] | [entidade] | [tipo, terminal, demanda, janela, ...] | [...] |
| [Barcaça] | [recurso móvel] | [capacidade, estoque, posição, estado] | [...] |
| [...] | [...] | [...] | [...] |

| Elemento | Tipo | Atributos | Quantidade / capacidade |
|:--|:--|:--|:--|
| Terminal do porto | Recurso | `id_terminal`, classe atendida, *c_j* berços | *M* terminais, *c_j* berços cada |
| Fundeio | Fila | Uma fila FIFO por terminal, sem desistência | Ilimitada |
| Navio | Entidade temporária | `id_navio`, classe (= terminal), instantes de notificação, chegada, manobras e operação | Gerado pelo processo de demanda |
| Pedido de combustível | Atributo do navio | Comprador (0/1), volume *a* (t), janela [`t_ini_operacao`, `t_fim_operacao`], estado (na carteira, programado, atendido, perdido), barcaça alocada | No máximo um por navio |
| Barcaça | Recurso móvel | Capacidade *L*, carga atual, posição, estado, vazão de bombeamento *pf*, velocidade *v*, rota | 6 × 3.000 t (até 9 nas alternativas) |
| Base BVE | Local | Berços de carga *B*, vazão de carga *q*, tempo de manobra, fila FIFO de barcaças | 1 base; 2 berços (3 ou 4 nas alternativas); estoque ilimitado |
| Canal | Rede | Distâncias *u_ij* entre a base e os terminais | Sem restrição de tráfego |
| PCO | Controlador | Instantes de programação, horizonte *H*, regra de prioridade, carteira de pedidos, programa vigente | 1 |

**Estados da barcaça:** `ociosa`, `em fila na base`, `carregando` (inclui as manobras na base), `navegando`, `aguardando janela`, `conectando`, `bombeando`, `desconectando`, `fora de operação`.


## 2. Processos

> Especifique os quatro processos e, principalmente, o modo como eles se encontram: quem espera por quem, e o que acontece nessa espera. Um fluxograma por processo costuma ser mais eficiente que a descrição em texto.
>
> - **Demanda (navios):** comunicação prévia, chegada, atracação, operação portuária, abastecimento, desatracação e saída.
> - **Abastecimento (barcaças):** deslocamento pelo canal, atracação junto ao navio, bombeamento, desatracação e destino seguinte.
> - **Carregamento (terminal BVE):** chegada à base, espera por berço, carregamento e liberação.
> - **Coordenação (PCO):** recebimento dos pedidos, programação e despacho das barcaças.

### 2.1 Demanda: navios (submodelo 1)

Segue o modelo de referência. Cada terminal *j* tem um gerador independente: as notificações seguem um processo de Poisson de taxa *λ_j*. Na notificação sorteiam-se a condição de comprador (Bernoulli com `prob_compra_j`) e, se for o caso, o volume. O navio chega ao fundeio após a antecedência (48 h, constante), espera berço livre (FIFO), faz a manobra de entrada (o berço é ocupado no início dela), opera no terminal, faz a manobra de saída e libera o berço. A operação no terminal é a **janela de atendimento**. Cada terminal funciona como uma fila M/G/*c_j*.

```mermaid
flowchart LR
    A([Notificação: navio e demanda]) --> B[Chegada ao fundeio]
    B --> C{Berço livre no terminal?}
    C -- não --> D[Espera no fundeio - FIFO] --> C
    C -- sim --> E[Manobra de entrada]
    E --> F[Operação no terminal = janela de atendimento]
    F --> G[Manobra de saída] --> H([Partida])
```

```text
processo gerador_navios(j):
    repetir enquanto t < horizonte:
        aguardar ~ Exponencial(taxa = lambda_j)
        criar navio i da classe j; t_notificacao[i] = t
        comprador[i] = Bernoulli(prob_compra_j)
        volume[i]    = dist_volume_j se comprador[i], senão 0
        iniciar navio(i, j)

processo navio(i, j):
    aguardar antecedencia_j;           t_chegada[i] = t
    requisitar berço do terminal j;    t_ini_manobra[i] = t
    aguardar tempo_manobra_j;          t_ini_operacao[i] = t
    aguardar ~ dist_operacao_j;        t_fim_operacao[i] = t
    aguardar tempo_manobra_j;          t_fim_manobra[i] = t
    liberar berço; gravar navio i em log_navios
```

### 2.2 Suprimento: barcaças e base (submodelo 2)

As barcaças operam em ciclo contínuo: carga na base, navegação, abastecimento e retorno. Neste submodelo não há navios nem PCO: o destino e o volume de cada viagem são sorteados, o que permite verificar a frota e a base isoladamente. O destino é o terminal *j*, com probabilidade proporcional a *λ_j* · `prob_compra_j`, e o volume segue `dist_volume_j`. Cada viagem faz um atendimento.

Na base, a barcaça entra na fila única, ocupa um berço, faz a manobra, completa a carga até *L* (tempo = volume faltante / *q*), faz a manobra de saída e libera o berço.

```mermaid
flowchart LR
    A{Berço da base livre?} -- não --> B[Fila na base - FIFO] --> A
    A -- sim --> C[Manobra e carregamento até L]
    C --> D[Sorteio de destino e volume]
    D --> E[Navegação até o terminal]
    E --> F[Conexão, bombeamento e desconexão]
    F --> G[Retorno à base] --> A
```

```text
processo barcaca(k):
    repetir enquanto t < horizonte:
        requisitar berço da base                      # fila FIFO
        aguardar tempo_manobra_base
        aguardar (L_k - carga_k) / q;  carga_k = L_k
        aguardar tempo_manobra_base;   liberar berço
        sortear terminal j e volume a
        aguardar u(base, j) / v_k
        aguardar t_conexao; aguardar a / pf_k; aguardar t_desconexao
        carga_k = carga_k - a
        aguardar u(j, base) / v_k
        gravar estados de k em log_barcacas
```

### 2.3 Coordenação: PCO (submodelo 3)

O submodelo integrado substitui o sorteio do submodelo 2 pelos pedidos reais. Cada navio comprador de `log_navios.csv` entra na carteira do PCO no instante da notificação. Nos instantes de programação e nos eventos de reprogramação, o PCO aloca pedidos às barcaças e publica as rotas. A barcaça pode atender vários navios em sequência enquanto tiver carga, e ao fim da rota retorna à base.

```mermaid
flowchart LR
    subgraph Navios
        N1[Notificação do comprador]
        N2[Operação no terminal]
    end
    subgraph PCO
        P1[Carteira de pedidos] --> P2[Programação 09:30 e 16:30 ou reprogramação] --> P3[Rotas das barcaças]
    end
    subgraph Barcaças
        B1[Navegação até o navio] --> B2{Atendimento cabe na janela?}
        B2 -- sim --> B3[Conexão, bombeamento e desconexão] --> B4{Há próximo pedido na rota?}
        B2 -- não --> B5[Pedido perdido] --> B4
        B4 -- sim --> B1
        B4 -- não --> B6[Retorno e carregamento na base]
    end
    N1 --> P1
    P3 --> B1
    N2 -. janela .-> B2
    B6 -. barcaça disponível .-> P2
```

## 3. Regras de decisão do PCO

> Esta seção define o comportamento do sistema e costuma ser a mais negligenciada. Especifique:
>
> - como um pedido de abastecimento é aceito e programado;
> - qual barcaça é alocada a qual navio, e segundo que critério;
> - o que a barcaça faz ao terminar um atendimento (retornar à base, atender outro navio, ir ao fundeio);
> - como são tratados os conflitos: dois navios elegíveis, nenhuma barcaça disponível, janela em risco.
>
> Escreva as regras de forma que possam ser traduzidas em código, em texto estruturado ou em pseudocódigo.

## 3. Regras de decisão do PCO

Regra do cenário-base, inspirada na prática de operação análoga (Costa e Mesquita, 2023): ordenação pelo fim da janela e alocação à barcaça que conclui primeiro.

```text
EM cada instante de programação t (09:30 e 16:30) OU evento de reprogramação:
    P <- pedidos na carteira, não iniciados, com t_ini_operacao <= t + H      # H = 36 h
    ORDENAR P por menor t_fim_operacao; desempate: maior volume
    PARA cada pedido j em P:
        PARA cada barcaça k fora do estado "fora de operação":
            SE carga prevista de k ao fim da rota < a_j:
                incluir ida à base e recarga antes de j
            C_jk <- max(disponível_k + deslocamento, t_ini_operacao_j)
                    + t_conexao + a_j / pf_k + t_desconexao
        K* <- barcaças com C_jk <= t_fim_operacao_j
        SE K* vazio: j permanece na carteira
        SENÃO: k* <- argmin C_jk em K*   (desempate: menor deslocamento)
               anexar j ao fim da rota de k*
```

- **Início do atendimento:** ao chegar ao navio, a barcaça só conecta se *t* + *t_conexão* + *a*/*pf* + *t_desconexão* ≤ `t_fim_operacao`. Caso contrário, o pedido é perdido e a barcaça segue a rota. Não há bombeamento parcial.
- **Chegada antecipada:** se a barcaça chega antes de `t_ini_operacao`, aguarda a janela.
- **Fim do atendimento:** a barcaça segue para o próximo pedido da rota se tiver carga suficiente; senão, ou se a rota acabou, retorna à base e recarrega.
- **Dois navios elegíveis para a mesma barcaça:** vale a ordenação acima.
- **Nenhuma barcaça disponível:** o pedido fica na carteira; é perdido quando `t_fim_operacao` passa sem atendimento iniciado.
- **Reprogramação:** disparada por falha de barcaça. Pedidos em atendimento não são realocados.

Na alternativa A4 variam-se o intervalo de programação (inclusive programação a cada evento), o horizonte *H*, o critério de ordenação e o agrupamento de pedidos por viagem.


## 4. Eventos e variáveis de estado

> Relacione os eventos que alteram o estado do sistema e o efeito de cada um. As variáveis de estado são o que o modelo precisa "lembrar" entre eventos: estoque das barcaças, ocupação dos berços, fila de pedidos, posição de cada barcaça.

| Evento | Disparado por | Efeito no estado |
|:--|:--|:--|
| [chegada de navio] | [processo de geração] | [...] |
| [...] | [...] | [...] |

## 4. Eventos e variáveis de estado

| Submodelo | Evento | Efeito no estado |
|:--|:--|:--|
| 1 | Notificação | Cria o navio e sorteia a demanda; se comprador, o pedido entra na carteira (submodelo 3) |
| 1 | Chegada ao fundeio | Incrementa *N_j*; navio entra na fila do terminal |
| 1 | Início da manobra de entrada | Ocupa um berço do terminal; navio deixa a fila |
| 1 | Início da operação | Abre a janela de atendimento |
| 1 | Fim da operação | Fecha a janela; pedido não iniciado passa a perdido |
| 1 | Fim da manobra de saída | Libera o berço; decrementa *N_j*; navio deixa o sistema |
| 2, 3 | Chegada à base | Barcaça `em fila na base` ou `carregando` |
| 2, 3 | Início / fim do carregamento | Berço da base ocupado / liberado; carga = *L* |
| 2, 3 | Saída da base | Barcaça `navegando` |
| 2, 3 | Chegada ao terminal | Barcaça `aguardando janela`, `conectando` ou segue (pedido perdido) |
| 2, 3 | Início do bombeamento | Barcaça `bombeando` |
| 2, 3 | Fim do atendimento | Carga reduzida em *a*; pedido atendido |
| 2, 3 | Falha / retorno da barcaça | Barcaça `fora de operação` / disponível; reprogramação |
| 3 | Instante de programação | Rotas das barcaças atualizadas |

**Variáveis de estado:** *N_j(t)*, número de navios da classe *j* no porto (ocupando berço: min(*N_j*, *c_j*); em espera: max(0, *N_j* − *c_j*)); estado, posição, carga e rota de cada barcaça; ocupação e fila dos berços da base; carteira de pedidos e seus estados; programa vigente.


## 5. Parâmetros e variáveis aleatórias

> Distinga o que é constante do que é aleatório. Para as variáveis aleatórias, indique o que deverá ser estimado a partir dos dados.

| Parâmetro / variável | Símbolo | Unidade | Natureza |
|:--|:--|:--|:--|
| [...] | [...] | [...] | [...] |

**Parâmetros do cenário-base**

| Parâmetro | Símbolo | Unidade | Valor / natureza | Origem |
|:--|:--|:--|:--|:--|
| Número de barcaças / capacidade | *m* / *L* | un / t | 6 / 3.000, constantes | Carta-convite |
| Berços da base | *B* | un | 2, constante | Carta-convite |
| Vazão de carga por berço | *q* | t/h | 400, constante | Premissa (Costa e Mesquita, 2023) |
| Vazão de bombeamento | *pf* | t/h | 200, constante | Premissa (idem) |
| Velocidade | *v* | km/h | 4,5, constante | Premissa (idem) |
| Conexão / desconexão | *t_c*, *t_d* | h | Aleatórias, média 1,5 | Premissa (idem); estimar |
| Manobra na base | *t_mb* | h | Aleatória | Estimar |
| Taxa de chegada por terminal | *λ_j* | navios/h | Poisson | Estimar |
| Berços por terminal | *c_j* | un | Constante | Solicitar |
| Antecedência | - | h | 48, constante | Modelo de referência |
| Manobra no terminal | - | h | Constante por classe | Estimar |
| Tempo de operação | - | h | Aleatório, por classe | Estimar |
| Probabilidade de compra | `prob_compra_j` | - | Bernoulli, por classe | Estimar |
| Volume demandado | *a* | t | Aleatório, por classe | Estimar |
| Distâncias | *u_ij* | km | Constantes | Solicitar |
| Tempo entre falhas / duração da falha | *TEF* / *TR* | h | Aleatórios | Estimar, se houver registros |
| Programação / horizonte | - / *H* | hh:mm / h | 09:30 e 16:30 / 36 | Premissa (Costa e Mesquita, 2023) |
| Horizonte de simulação | - | dias | 500 | Modelo de referência |
| Fator de demanda | *f_d* | - | 1,0 a 1,4 (fator de experimento) | Proposta |

Os itens a estimar e a solicitar constam de G112_Dados.





## 6. Indicadores

> Para cada indicador da proposta, mostre como ele será calculado a partir dos eventos registrados na simulação. Exemplo do que se espera: tempo de espera é a diferença entre o instante de início do atendimento e o instante de chegada. Indicadores de utilização precisam declarar quais estados contam como recurso ocupado.

| Indicador | Cálculo a partir dos eventos | Unidade |
|:--|:--|:--|
| [...] | [...] | [...] |

| Indicador | Cálculo a partir dos logs | Unidade |
|:--|:--|:--|
| Nível de serviço | Pedidos com `status` = atendido ÷ pedidos | % |
| Pedidos perdidos | 1 − nível de serviço | % |
| Volume entregue / volume perdido | Soma de `volume_t` atendidos / perdidos ÷ meses simulados | t/mês |
| Capacidade | Maior *f_d* com nível de serviço ≥ referência, vezes o volume demandado no cenário-base | t/mês |
| Espera por berço na base | Início de `carregando` − início de `em fila na base` | h |
| Utilização da barcaça | Tempo fora de `ociosa` e `fora de operação` ÷ tempo fora de `fora de operação` | % |
| Utilização dos berços da base | Tempo total em `carregando` ÷ (*B* × tempo simulado) | % |
| Tempo de ciclo da barcaça | Saída da base − saída anterior da base | h |
| Custo anual e custo por t adicional | Custo anual equivalente da alternativa; ÷ (capacidade − capacidade do cenário-base) | R$/ano; R$/t |
| Indicadores dos navios (validação da demanda) | Espera no fundeio, permanência, duração da janela e utilização dos terminais, conforme o modelo de referência | h, % |

As estatísticas são coletadas após o período de aquecimento e reportadas com intervalo de confiança de 95% entre replicações.

## 7. Premissas e questões em aberto

> **Premissas:** registre as principais hipóteses de modelagem adotadas e sua justificativa.
>
> **Questões em aberto:** decisões de modelagem ainda não resolvidas e informações que faltam. As que dependem de dados alimentam o documento Gxxx_Dados.

| Premissa | Justificativa |
|:--|:--|
| O navio deixa o porto ao fim da operação, abastecido ou não | Modelo de referência; separa demanda e suprimento |
| Antecedência constante; fila FIFO por terminal, sem desistência | Modelo de referência |
| O PCO conhece a janela real (`t_ini_operacao`, `t_fim_operacao`) a partir da notificação | O log dos navios entrega a janela ao PCO; o efeito de janelas estimadas será tratado na sensibilidade |
| Um pedido é atendido por uma única barcaça; volume ≤ *L*; sem bombeamento parcial | Simplicidade; operação análoga |
| O fator de demanda multiplica `prob_compra_j` (limitado a 1) | O crescimento de 20% é da BVE, não do tráfego do porto |
| No submodelo 2, destino proporcional a *λ_j* · `prob_compra_j` e um atendimento por viagem | Reproduz a distribuição esperada dos pedidos sem o PCO |
| Estoque ilimitado na base; carga completada até *L* | Suprimento fora do escopo |
| Operação 24 h, sem restrição de tráfego ou clima | Ausência de dados; tratado na sensibilidade |
| Tempos e vazões de Costa e Mesquita (2023) até a chegada dos dados | Única referência disponível |

| Questão em aberto | Destino |
|:--|:--|
| O crescimento da demanda virá de mais navios ou de mais compradores? | G112_Dados, Demanda |
| Em que momento o PCO conhece a janela de cada navio? | G112_Dados, Coordenação |
| Regra real do PCO e frequência de reprogramação | G112_Dados, Coordenação |
| Uma barcaça pode dividir um pedido ou atender dois navios ao mesmo tempo? | G112_Dados, Abastecimento |
| A barcaça recarrega sempre até a capacidade? Há limite de estoque ou de horário na base? | G112_Dados, Carregamento |
| Nível de serviço de referência | G112_Dados, Demanda |